In [1]:
!pip install --quiet --upgrade google-cloud-pubsub google-cloud-bigquery

import time
from google.cloud import bigquery, pubsub_v1
from google.api_core import exceptions as gexc

PROJECT_ID      = "qwiklabs-gcp-00-c521a9ba0b6e"
DATASET_ID      = "flight_data"
TABLE_ID        = "flight_transponder_msgs"
SUBSCRIPTION_ID = "flight-transponder-sub"

PUBLISHER_PROJECT = "paul-leroy"
TOPIC_NAME        = "flight-transponder"
TOPIC_PATH        = f"projects/{PUBLISHER_PROJECT}/topics/{TOPIC_NAME}"

bq = bigquery.Client(project=PROJECT_ID)
TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"
print("Clients ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.8/324.8 kB 7.4 MB/s eta 0:00:00
Clients ready.


In [2]:
ds = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
ds.location = "US"
bq.create_dataset(ds, exists_ok=True)

schema = [
    bigquery.SchemaField("MT",   "STRING",  mode="NULLABLE", description="SEL ID AIR STA CLK MSG info"),
    bigquery.SchemaField("TT",   "INT64",   mode="NULLABLE", description="1 - 8"),
    bigquery.SchemaField("SID",  "STRING",  mode="NULLABLE", description="Database Session record number"),
    bigquery.SchemaField("AID",  "STRING",  mode="NULLABLE", description="Database Aircraft record number"),
    bigquery.SchemaField("Hex",  "STRING",  mode="NULLABLE", description="Aircraft Mode S hexadecimal code"),
    bigquery.SchemaField("FID",  "STRING",  mode="NULLABLE", description="Database Flight record number"),
    bigquery.SchemaField("DMG",  "DATE",    mode="NULLABLE", description="Date message generated"),
    bigquery.SchemaField("TMG",  "TIME",    mode="NULLABLE", description="Time message generated"),
    bigquery.SchemaField("DML",  "DATE",    mode="NULLABLE", description="Date message logged"),
    bigquery.SchemaField("TML",  "TIME",    mode="NULLABLE", description="Time message logged"),
    bigquery.SchemaField("CS",   "STRING",  mode="NULLABLE", description="Callsign (flight number or registration)"),
    bigquery.SchemaField("Alt",  "INT64",   mode="NULLABLE", description="Mode C altitude (Flight Level)"),
    bigquery.SchemaField("GS",   "INT64",   mode="NULLABLE", description="Ground Speed"),
    bigquery.SchemaField("Trk",  "INT64",   mode="NULLABLE", description="Track"),
    bigquery.SchemaField("Lat",  "FLOAT64", mode="NULLABLE", description="Latitude (N/E positive, S/W negative)"),
    bigquery.SchemaField("Lng",  "FLOAT64", mode="NULLABLE", description="Longitude (N/E positive, S/W negative)"),
    bigquery.SchemaField("VR",   "INT64",   mode="NULLABLE", description="Vertical Rate"),
    bigquery.SchemaField("Sq",   "STRING",  mode="NULLABLE", description="Assigned Mode A squawk code"),
    bigquery.SchemaField("Alrt", "INT64",   mode="NULLABLE", description="Flag to indicate squawk has changed"),
    bigquery.SchemaField("Emer", "INT64",   mode="NULLABLE", description="Flag to indicate emergency code has been set"),
    bigquery.SchemaField("SPI",  "INT64",   mode="NULLABLE", description="Flag to indicate transponder Ident has been activated"),
    bigquery.SchemaField("Gnd",  "INT64",   mode="NULLABLE", description="Flag to indicate ground squat switch is active"),
]

table = bigquery.Table(TABLE_REF, schema=schema)
bq.create_table(table, exists_ok=True)
print(f"Table `{TABLE_ID}` ready with {len(schema)} fields.")

Table `flight_transponder_msgs` ready with 22 fields.


In [3]:
subscriber = pubsub_v1.SubscriberClient()
subscription_path = subscriber.subscription_path(PROJECT_ID, SUBSCRIPTION_ID)

try:
    subscriber.create_subscription(
        request={"name": subscription_path, "topic": TOPIC_PATH}
    )
    print(f"Subscription created: {subscription_path}")
except gexc.AlreadyExists:
    print("Subscription already exists — reusing it.")

Subscription created: projects/qwiklabs-gcp-00-c521a9ba0b6e/subscriptions/flight-transponder-sub


In [4]:
SCHEMA_FIELDS = [
    ("MT","STRING"),("TT","INT64"),("SID","STRING"),("AID","STRING"),
    ("Hex","STRING"),("FID","STRING"),("DMG","DATE"),("TMG","TIME"),
    ("DML","DATE"),("TML","TIME"),("CS","STRING"),("Alt","INT64"),
    ("GS","INT64"),("Trk","INT64"),("Lat","FLOAT64"),("Lng","FLOAT64"),
    ("VR","INT64"),("Sq","STRING"),("Alrt","INT64"),("Emer","INT64"),
    ("SPI","INT64"),("Gnd","INT64"),
]

def parse_line(line):
    parts = line.split(",")
    if len(parts) < 22:
        parts += [""] * (22 - len(parts))
    parts = parts[:22]

    row = {}
    for (name, ftype), raw in zip(SCHEMA_FIELDS, parts):
        val = raw.strip()
        if val == "":
            row[name] = None
        elif ftype == "INT64":
            try: row[name] = int(val)
            except ValueError: row[name] = None
        elif ftype == "FLOAT64":
            try: row[name] = float(val)
            except ValueError: row[name] = None
        elif ftype == "DATE":
            row[name] = val.replace("/", "-")
        else:
            row[name] = val
    return row

sample = "MSG,3,1,1,ACAE7C,1,2025/04/15,00:43:07.127,2025/04/15,00:43:07.171,,2350,,,33.59715,-117.64749,,,0,,0,0"
parse_line(sample)

{'MT': 'MSG',
 'TT': 3,
 'SID': '1',
 'AID': '1',
 'Hex': 'ACAE7C',
 'FID': '1',
 'DMG': '2025-04-15',
 'TMG': '00:43:07.127',
 'DML': '2025-04-15',
 'TML': '00:43:07.171',
 'CS': None,
 'Alt': 2350,
 'GS': None,
 'Trk': None,
 'Lat': 33.59715,
 'Lng': -117.64749,
 'VR': None,
 'Sq': None,
 'Alrt': 0,
 'Emer': None,
 'SPI': 0,
 'Gnd': 0}

In [5]:
RUN_SECONDS = 180

deadline = time.time() + RUN_SECONDS
total_inserted = 0
print(f"Streaming for {RUN_SECONDS}s...")

while time.time() < deadline:
    try:
        resp = subscriber.pull(
            request={"subscription": subscription_path, "max_messages": 200},
            timeout=10,
        )
    except gexc.DeadlineExceeded:
        continue

    if not resp.received_messages:
        continue

    rows, ack_ids = [], []
    for rm in resp.received_messages:
        text = rm.message.data.decode("utf-8")
        for line in text.splitlines():
            line = line.strip()
            if line:
                rows.append(parse_line(line))
        ack_ids.append(rm.ack_id)

    if rows:
        errors = bq.insert_rows_json(TABLE_REF, rows)
        if errors:
            print("Insert errors:", errors[:2])
        else:
            total_inserted += len(rows)

    subscriber.acknowledge(
        request={"subscription": subscription_path, "ack_ids": ack_ids}
    )
    print(f"  inserted so far: {total_inserted}", end="\r")

print(f"\nDone. Total rows inserted: {total_inserted}")

Streaming for 180s...
  inserted so far: 69955
Done. Total rows inserted: 69955


In [6]:
bq.query(f"""
    SELECT COUNT(*) AS record_count
    FROM `{TABLE_REF}`
""").to_dataframe()

,record_count
0,69955


In [8]:
geo_sql = f"""
SELECT
    ST_GEOGPOINT(Lng, Lat) AS Location
FROM `{TABLE_REF}`
WHERE Lat IS NOT NULL
  AND Lng IS NOT NULL
"""
print(geo_sql)

# Preview that points exist before opening GeoViz
bq.query(geo_sql + " LIMIT 10").to_dataframe()


SELECT
    ST_GEOGPOINT(Lng, Lat) AS Location
FROM `qwiklabs-gcp-00-c521a9ba0b6e.flight_data.flight_transponder_msgs`
WHERE Lat IS NOT NULL
  AND Lng IS NOT NULL



,Location
0,POINT(-0.48271 51.9034)
1,POINT(-1.98481 50.70973)
2,POINT(-0.22863 51.31389)
3,POINT(-0.75623 51.45372)
4,POINT(-1.26663 50.89156)
5,POINT(-0.64758 51.5094)
6,POINT(-0.49789 51.8969)
7,POINT(-0.23346 51.46541)
8,POINT(-0.69038 51.64445)
9,POINT(-0.61316 51.32625)
